In [ ]:
"""
analyse_dataset.py
------------------
Reads device_compromise_dataset.xlsx and prints LaTeX-ready statistical

Requires: pandas, openpyxl, matplotlib, numpy
    pip install pandas openpyxl matplotlib numpy
"""

import os
import textwrap
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────

DATASET   = "device_compromise_dataset.xlsx"
PLOT_DIR  = "analysis_plots"
os.makedirs(PLOT_DIR, exist_ok=True)

TACTIC_ORDER = [
    "Initial Access", "Credential Access", "Defense Evasion",
    "Privilege Escalation", "Persistence", "Discovery",
    "Lateral Movement", "Collection",
]

SEVERITY_ORDER  = ["CRITICAL", "HIGH", "MEDIUM", "LOW"]
SEVERITY_COLORS = {
    "CRITICAL": "#d62728", "HIGH": "#ff7f0e",
    "MEDIUM":   "#1f77b4", "LOW":  "#2ca02c",
}

# ─────────────────────────────────────────────────────────────────────────────
# LOAD & CLEAN
# ─────────────────────────────────────────────────────────────────────────────

df = pd.read_excel(DATASET, index_col=False)
df = df.fillna("")
df["tactics"]   = df["tactics"].astype(str).str.strip()
df["baseScore"]  = pd.to_numeric(df["baseScore"],  errors="coerce")
df["exploitabilityScore"] = pd.to_numeric(df["exploitabilityScore"], errors="coerce")

# Per-CVE view (one row per unique CVE, using first occurrence)
cve_df = (
    df[df["CVE ID"] != ""]
    .drop_duplicates("CVE ID")
    .set_index("CVE ID")
)

n_cves      = len(cve_df)
n_tactics   = len([t for t in df["tactics"].unique() if t])
n_cwes_raw  = len([c for c in df["cwe_Name"].unique() if c])

# Deduplicated CWE names (merge deprecated aliases)
CWE_ALIAS = {
    "Use of Hard-coded Password":        "Use of Hard-coded Credentials (CWE-798)",
    "Use of Hard-coded Credentials":     "Use of Hard-coded Credentials (CWE-798)",
    "DEPRECATED: Authentication Bypass Issues": "Improper Authentication (CWE-287)",
}
df["cwe_clean"] = df["cwe_Name"].map(lambda x: CWE_ALIAS.get(x, x))

# Unique CVE-CWE pairs (deduplicated)
cwe_cve = (
    df[df["cwe_Name"] != ""]
    .drop_duplicates(["CVE ID", "cwe_clean"])
)

# Unique CVE-tactic pairs
tac_cve = (
    df[df["tactics"] != ""]
    .drop_duplicates(["CVE ID", "tactics"])
)

# Unique attack paths (tactic × CWE × technique × subtechnique × CAPEC)
paths_df = (
    df[df["tactics"] != ""]
    .drop_duplicates(["tactics", "cwe_Name", "technique_name",
                      "subtechnique_name", "capec_Name"])
)

# ─────────────────────────────────────────────────────────────────────────────
# HELPER: pretty print a section
# ─────────────────────────────────────────────────────────────────────────────

def section(title):
    print(f"\n{'═'*70}")
    print(f"  {title}")
    print(f"{'═'*70}")


def bullet(label, value):
    print(f"  • {label:<52} {value}")


# ─────────────────────────────────────────────────────────────────────────────
# 1.  HIGH-LEVEL SUMMARY
# ─────────────────────────────────────────────────────────────────────────────

section("1. HIGH-LEVEL DATASET SUMMARY")

scores        = cve_df["baseScore"].dropna()
n_critical    = (cve_df["baseSeverity"] == "CRITICAL").sum()
n_network     = (cve_df["attackVector"] == "NETWORK").sum()
n_physical    = (cve_df["attackVector"] == "PHYSICAL").sum()
n_low_complex = (cve_df["attackComplexity"] == "LOW").sum()
n_no_priv     = (cve_df["privilegesRequired"] == "NONE").sum()
n_no_ui       = (cve_df["userInteraction"] == "NONE").sum()

bullet("Total CVEs",                                    n_cves)
bullet("MITRE ATT&CK tactics covered",                  n_tactics)
bullet("Unique weakness classes (raw CWE names)",       n_cwes_raw)
bullet("Unique weakness classes (deduplicated)",
       cwe_cve["cwe_clean"].nunique())
bullet("CVSS score — mean",                             f"{scores.mean():.2f}")
bullet("CVSS score — median",                           f"{scores.median():.2f}")
bullet("CVSS score — range",
       f"{scores.min():.1f} – {scores.max():.1f}")
bullet("Critical severity (CVSS ≥ 9.0)",
       f"{n_critical} / {n_cves}  ({100*n_critical/n_cves:.0f}%)")
bullet("Network-exploitable",
       f"{n_network} / {n_cves}  ({100*n_network/n_cves:.0f}%)")
bullet("Physical access only",
       f"{n_physical} / {n_cves}  ({100*n_physical/n_cves:.0f}%)")
bullet("Attack complexity LOW (all CVEs)",
       f"{n_low_complex} / {n_cves}  ({100*n_low_complex/n_cves:.0f}%)")
bullet("No privileges required (all CVEs)",
       f"{n_no_priv} / {n_cves}  ({100*n_no_priv/n_cves:.0f}%)")
bullet("No user interaction required (all CVEs)",
       f"{n_no_ui} / {n_cves}  ({100*n_no_ui/n_cves:.0f}%)")

# ─────────────────────────────────────────────────────────────────────────────
# 2.  IMPACT ANALYSIS
# ─────────────────────────────────────────────────────────────────────────────

section("2. IMPACT ANALYSIS (CIA TRIAD)")

for col, label in [
    ("confidentialityImpact", "Confidentiality HIGH"),
    ("integrityImpact",       "Integrity HIGH"),
    ("availabilityImpact",    "Availability HIGH"),
]:
    n_high = (cve_df[col] == "HIGH").sum()
    bullet(label, f"{n_high} / {n_cves}  ({100*n_high/n_cves:.0f}%)")

print()
print("  Per-CVE impact summary:")
print(f"  {'CVE ID':<20} {'Score':>6}  {'Severity':<10} "
      f"{'Conf':>6}  {'Integ':>6}  {'Avail':>6}  {'AV':<10}")
print("  " + "-"*72)
for cid, row in cve_df.iterrows():
    print(f"  {cid:<20} {row['baseScore']:>6.1f}  {row['baseSeverity']:<10} "
          f"{row['confidentialityImpact']:>6}  {row['integrityImpact']:>6}  "
          f"{row['availabilityImpact']:>6}  {row['attackVector']:<10}")

# ─────────────────────────────────────────────────────────────────────────────
# 3.  TACTIC DISTRIBUTION
# ─────────────────────────────────────────────────────────────────────────────

section("3. TACTIC DISTRIBUTION")

tac_cve_count = tac_cve.groupby("tactics")["CVE ID"].count().reindex(
    TACTIC_ORDER).fillna(0).astype(int)
paths_per_tac = paths_df.groupby("tactics").size().reindex(
    TACTIC_ORDER).fillna(0).astype(int)

print(f"\n  {'Tactic':<35} {'CVEs':>5}  {'Attack Paths':>13}")
print("  " + "-"*56)
for tac in TACTIC_ORDER:
    print(f"  {tac:<35} {tac_cve_count.get(tac,0):>5}  "
          f"{paths_per_tac.get(tac,0):>13}")

tactics_per_cve = tac_cve.groupby("CVE ID")["tactics"].count().sort_values(
    ascending=False)
print(f"\n  Average tactics per CVE: {tactics_per_cve.mean():.1f}")
print(f"  Max tactics per CVE:     {tactics_per_cve.max()}  "
      f"({tactics_per_cve.idxmax()})")
print(f"  Min tactics per CVE:     {tactics_per_cve.min()}  "
      f"({tactics_per_cve.idxmin()})")

print("\n  Tactics per CVE breakdown:")
for cid, n in tactics_per_cve.items():
    tacs = sorted(tac_cve[tac_cve["CVE ID"]==cid]["tactics"].tolist())
    print(f"    {cid:<20} {n} tactics: {', '.join(tacs)}")

# ─────────────────────────────────────────────────────────────────────────────
# 4.  CWE ANALYSIS
# ─────────────────────────────────────────────────────────────────────────────

section("4. CWE / WEAKNESS ANALYSIS")

cwe_count = cwe_cve.groupby("cwe_clean")["CVE ID"].count().sort_values(
    ascending=False)
total_cwe_mappings = cwe_count.sum()

print(f"\n  Total CVE→CWE mappings: {total_cwe_mappings}")
print(f"\n  {'Weakness':<58} {'CVEs':>5}  {'%':>6}")
print("  " + "-"*70)
for cwe, cnt in cwe_count.items():
    short = textwrap.shorten(cwe, width=56, placeholder="…")
    print(f"  {short:<58} {cnt:>5}  {100*cnt/n_cves:>5.0f}%")

dominant_cwe = cwe_count.idxmax()
dominant_pct = 100 * cwe_count.max() / n_cves
print(f"\n  Dominant weakness: {dominant_cwe}")
print(f"  Appears in {cwe_count.max()} CVEs ({dominant_pct:.0f}% of dataset)")

# ─────────────────────────────────────────────────────────────────────────────
# 5.  TECHNIQUE & CAPEC ANALYSIS
# ─────────────────────────────────────────────────────────────────────────────

section("5. TECHNIQUE & CAPEC ANALYSIS")

tech_cve = (
    df[df["technique_name"] != ""]
    .drop_duplicates(["CVE ID", "technique_name", "tactics"])
)
tech_count = tech_cve["technique_name"].value_counts()
n_unique_techs = tech_count.shape[0]
print(f"\n  Unique techniques: {n_unique_techs}")
print(f"  Top-5 techniques:")
for tech, cnt in tech_count.head(5).items():
    print(f"    {tech:<50} {cnt:>3} occurrences")

cap_cve = df[df["capec_Name"] != ""].drop_duplicates(["CVE ID", "capec_Name"])
cap_count = cap_cve["capec_Name"].value_counts()
n_unique_capec = cap_count.shape[0]
print(f"\n  Unique CAPEC attack patterns: {n_unique_capec}")
print(f"  Top-5 CAPECs:")
for cap, cnt in cap_count.head(5).items():
    short = textwrap.shorten(cap, width=56, placeholder="…")
    print(f"    {short:<58} {cnt:>3}")

# CAPEC likelihood distribution
cap_like = df[df["capec_Likelihood_Of_Attack"]!=""]\
             ["capec_Likelihood_Of_Attack"].value_counts()
cap_sev  = df[df["capec_Typical_Severity"]!=""]\
             ["capec_Typical_Severity"].value_counts()

print(f"\n  CAPEC Likelihood of Attack:  {dict(cap_like)}")
print(f"  CAPEC Typical Severity:      {dict(cap_sev)}")

# CWE Likelihood of Exploit
cwe_like = df[df["cwe_Likelihood_Of_Exploit"]!=""]\
             ["cwe_Likelihood_Of_Exploit"].value_counts()
print(f"  CWE Likelihood of Exploit:   {dict(cwe_like)}")

# ─────────────────────────────────────────────────────────────────────────────
# 6.  ATTACK TREE STRUCTURAL METRICS
# ─────────────────────────────────────────────────────────────────────────────

section("6. ATTACK TREE STRUCTURAL METRICS")

print(f"\n  Total unique attack paths across all tactics: "
      f"{len(paths_df)}")

print(f"\n  {'Tactic':<35} {'Paths':>6}  {'CWEs':>5}  {'Techs':>6}  {'CAPECs':>7}")
print("  " + "-"*62)
for tac in TACTIC_ORDER:
    tdf = paths_df[paths_df["tactics"]==tac]
    if tdf.empty:
        continue
    n_p = len(tdf)
    n_c = tdf["cwe_Name"].nunique()
    n_t = tdf["technique_name"].nunique()
    n_a = tdf["capec_Name"].nunique()
    print(f"  {tac:<35} {n_p:>6}  {n_c:>5}  {n_t:>6}  {n_a:>7}")

# Multi-tactic CVEs (breadth of exploitation)
print(f"\n  CVEs enabling ≥ 5 tactics (high-breadth vulnerabilities):")
for cid, n in tactics_per_cve[tactics_per_cve >= 5].items():
    print(f"    {cid}  →  {n} tactics")

# ─────────────────────────────────────────────────────────────────────────────
# 7.  LATEX TABLES (ready to paste)
# ─────────────────────────────────────────────────────────────────────────────

section("7. LATEX-READY TABLES")

# Table A: tactic summary
print("\n% ── TABLE A: Tactic distribution ──────────────────────────────")
print(r"\begin{table}[ht]")
print(r"\centering")
print(r"\caption{Distribution of attack paths across MITRE ATT\&CK tactics}")
print(r"\label{tab:tactic_distribution}")
print(r"\begin{tabular}{@{} l r r r @{}}")
print(r"\toprule")
print(r"\textbf{Tactic} & \textbf{CVEs} & \textbf{Paths} & \textbf{CWEs} \\")
print(r"\midrule")
for tac in TACTIC_ORDER:
    tdf = paths_df[paths_df["tactics"]==tac]
    n_p = len(tdf)
    n_c = tdf["cwe_Name"].nunique()
    tc  = tac_cve_count.get(tac, 0)
    if n_p > 0:
        print(f"{tac} & {tc} & {n_p} & {n_c} \\\\")
print(r"\bottomrule")
print(r"\end{tabular}")
print(r"\end{table}")

# Table B: CWE summary
print("\n% ── TABLE B: CWE distribution ──────────────────────────────────")
print(r"\begin{table}[ht]")
print(r"\centering")
print(r"\caption{Weakness classes identified in the dataset}")
print(r"\label{tab:cwe_distribution}")
print(r"\begin{tabularx}{\columnwidth}{@{} X r r @{}}")
print(r"\toprule")
print(r"\textbf{Weakness (CWE)} & \textbf{CVEs} & \textbf{\%} \\")
print(r"\midrule")
for cwe, cnt in cwe_count.items():
    safe = cwe.replace("&","\\&").replace("_","\\_")
    print(f"{safe} & {cnt} & {100*cnt/n_cves:.0f}\\% \\\\")
print(r"\bottomrule")
print(r"\end{tabularx}")
print(r"\end{table}")

# ─────────────────────────────────────────────────────────────────────────────
# 8.  FIGURES
# ─────────────────────────────────────────────────────────────────────────────

section("8. GENERATING FIGURES")

# ── Figure 1: CVSS score distribution per CVE ───────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
colors = [SEVERITY_COLORS.get(s, "#888") for s in cve_df["baseSeverity"]]
bars = ax.barh(cve_df.index, cve_df["baseScore"], color=colors,
               edgecolor="white", linewidth=0.5)
ax.set_xlabel("CVSS Base Score", fontsize=11)
ax.set_xlim(0, 11)
ax.axvline(9.0, color="red",    linestyle="--", linewidth=0.8,
           label="Critical threshold (9.0)")
ax.axvline(7.0, color="orange", linestyle="--", linewidth=0.8,
           label="High threshold (7.0)")
for bar, score in zip(bars, cve_df["baseScore"]):
    ax.text(score + 0.1, bar.get_y() + bar.get_height()/2,
            f"{score:.1f}", va="center", fontsize=9)
patches = [mpatches.Patch(color=v, label=k)
           for k, v in SEVERITY_COLORS.items()]
ax.legend(handles=patches + [
    plt.Line2D([0],[0], color="red",    linestyle="--", label="Critical (9.0)"),
    plt.Line2D([0],[0], color="orange", linestyle="--", label="High (7.0)"),
], fontsize=9, loc="lower right")
ax.set_title("CVSS Base Score per CVE", fontsize=12)
plt.tight_layout()
fig.savefig(f"{PLOT_DIR}/fig_cvss_scores.pdf", dpi=200, bbox_inches="tight")
fig.savefig(f"{PLOT_DIR}/fig_cvss_scores.png", dpi=200, bbox_inches="tight")
plt.close(fig)
print(f"  Saved: {PLOT_DIR}/fig_cvss_scores.pdf/.png")

# ── Figure 2: Tactic × Paths bar chart ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4.5))
ordered_tacs = [t for t in TACTIC_ORDER if t in paths_per_tac.index]
values  = [paths_per_tac.get(t, 0) for t in ordered_tacs]
cve_vals= [tac_cve_count.get(t, 0) for t in ordered_tacs]
x = range(len(ordered_tacs))
w = 0.38
b1 = ax.bar([i - w/2 for i in x], values,   width=w, label="Attack paths",
            color="#1f77b4", edgecolor="white")
b2 = ax.bar([i + w/2 for i in x], cve_vals, width=w, label="CVEs",
            color="#ff7f0e", edgecolor="white")
ax.set_xticks(list(x))
ax.set_xticklabels(ordered_tacs, rotation=30, ha="right", fontsize=9)
ax.set_ylabel("Count", fontsize=11)
ax.set_title("Attack Paths and CVEs per MITRE ATT\&CK Tactic", fontsize=12)
ax.legend(fontsize=10)
for bar in list(b1) + list(b2):
    h = bar.get_height()
    if h > 0:
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.15,
                str(int(h)), ha="center", va="bottom", fontsize=8)
plt.tight_layout()
fig.savefig(f"{PLOT_DIR}/fig_tactic_distribution.pdf", dpi=200,
            bbox_inches="tight")
fig.savefig(f"{PLOT_DIR}/fig_tactic_distribution.png", dpi=200,
            bbox_inches="tight")
plt.close(fig)
print(f"  Saved: {PLOT_DIR}/fig_tactic_distribution.pdf/.png")

# ── Figure 3: CWE distribution pie ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
short_labels = {
    "Use of Hard-coded Credentials (CWE-798)":              "Hard-coded Credentials\n(CWE-798)",
    "Improper Authentication (CWE-287)":                    "Improper Authentication\n(CWE-287)",
    "Cleartext Storage of Sensitive Information":           "Cleartext Storage\n(CWE-312)",
    "Cleartext Transmission of Sensitive Information":      "Cleartext Transmission\n(CWE-319)",
    "Incorrect Permission Assignment for Critical Resource":"Incorrect Permission\n(CWE-732)",
    "Operation on a Resource after Expiration or Release":  "Resource Expiration\n(CWE-672)",
}
labels = [short_labels.get(k, k) for k in cwe_count.index]
colors_pie = ["#d62728","#ff7f0e","#1f77b4","#2ca02c","#9467bd","#8c564b"]
wedges, texts, autotexts = ax.pie(
    cwe_count.values, labels=labels, autopct="%1.0f%%",
    colors=colors_pie[:len(cwe_count)],
    startangle=140, pctdistance=0.78,
    textprops={"fontsize": 9},
)
for at in autotexts:
    at.set_fontsize(8)
ax.set_title("Weakness Class Distribution (deduplicated)", fontsize=12)
plt.tight_layout()
fig.savefig(f"{PLOT_DIR}/fig_cwe_distribution.pdf", dpi=200,
            bbox_inches="tight")
fig.savefig(f"{PLOT_DIR}/fig_cwe_distribution.png", dpi=200,
            bbox_inches="tight")
plt.close(fig)
print(f"  Saved: {PLOT_DIR}/fig_cwe_distribution.pdf/.png")

# ── Figure 4: Heatmap — CVE × Tactic coverage ───────────────────────────────
all_cves = sorted(cve_df.index.tolist())
heat = pd.DataFrame(0, index=all_cves, columns=TACTIC_ORDER)
for _, row in tac_cve.iterrows():
    if row["CVE ID"] in heat.index and row["tactics"] in heat.columns:
        heat.loc[row["CVE ID"], row["tactics"]] = 1

fig, ax = plt.subplots(figsize=(11, 4.5))
im = ax.imshow(heat.values, aspect="auto", cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(TACTIC_ORDER)))
ax.set_xticklabels(TACTIC_ORDER, rotation=35, ha="right", fontsize=9)
ax.set_yticks(range(len(all_cves)))
ax.set_yticklabels(all_cves, fontsize=9)
ax.set_title("CVE – Tactic Coverage Matrix", fontsize=12)
for i in range(len(all_cves)):
    for j in range(len(TACTIC_ORDER)):
        if heat.values[i, j]:
            ax.text(j, i, "✓", ha="center", va="center",
                    fontsize=11, color="white", fontweight="bold")
plt.tight_layout()
fig.savefig(f"{PLOT_DIR}/fig_cve_tactic_heatmap.pdf", dpi=200,
            bbox_inches="tight")
fig.savefig(f"{PLOT_DIR}/fig_cve_tactic_heatmap.png", dpi=200,
            bbox_inches="tight")
plt.close(fig)
print(f"  Saved: {PLOT_DIR}/fig_cve_tactic_heatmap.pdf/.png")

# ── Figure 5: CAPEC Likelihood × Severity scatter ───────────────────────────
like_map = {"Low":1, "Medium":2, "High":3}
sev_map  = {"Low":1, "Medium":2, "High":3, "Very High":4}
cap_rows = df[(df["capec_Likelihood_Of_Attack"]!="") &
              (df["capec_Typical_Severity"]!="")].copy()
cap_rows["like_n"] = cap_rows["capec_Likelihood_Of_Attack"].map(like_map)
cap_rows["sev_n"]  = cap_rows["capec_Typical_Severity"].map(sev_map)
cap_rows = cap_rows.dropna(subset=["like_n","sev_n"])

fig, ax = plt.subplots(figsize=(6, 5))
jitter = np.random.RandomState(42)
jx = cap_rows["like_n"] + jitter.uniform(-0.12, 0.12, len(cap_rows))
jy = cap_rows["sev_n"]  + jitter.uniform(-0.12, 0.12, len(cap_rows))
ax.scatter(jx, jy, alpha=0.65, s=60, color="#1f77b4", edgecolors="white")
ax.set_xticks([1,2,3]);  ax.set_xticklabels(["Low","Medium","High"])
ax.set_yticks([1,2,3,4]);ax.set_yticklabels(["Low","Medium","High","Very High"])
ax.set_xlabel("CAPEC Likelihood of Attack", fontsize=11)
ax.set_ylabel("CAPEC Typical Severity",     fontsize=11)
ax.set_title("CAPEC Likelihood vs. Severity", fontsize=12)
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
fig.savefig(f"{PLOT_DIR}/fig_capec_likelihood_severity.pdf", dpi=200,
            bbox_inches="tight")
fig.savefig(f"{PLOT_DIR}/fig_capec_likelihood_severity.png", dpi=200,
            bbox_inches="tight")
plt.close(fig)
print(f"  Saved: {PLOT_DIR}/fig_capec_likelihood_severity.pdf/.png")

# ─────────────────────────────────────────────────────────────────────────────
# 9.  NARRATIVE INSIGHTS (paste-ready for LaTeX sections)
# ─────────────────────────────────────────────────────────────────────────────

section("9. NARRATIVE INSIGHTS FOR LATEX")

insights = f"""
────────────────────────────────────────────────────────────────────
RESULT ANALYSIS & DISCUSSION  →  Comparative Analysis
────────────────────────────────────────────────────────────────────

KEY NUMBERS TO USE IN TEXT:
  • {n_cves} CVEs → {n_tactics} MITRE ATT&CK tactics → {len(paths_df)} unique attack paths
  • {n_critical}/{n_cves} CVEs rated Critical (CVSS ≥ 9.0)
  • Mean CVSS: {scores.mean():.2f},  Median: {scores.median():.2f}
  • {n_network}/{n_cves} CVEs network-exploitable, {n_physical}/{n_cves} physical-only
  • ALL 10 CVEs: attack complexity LOW, no privileges required, no user interaction
  • Confidentiality HIGH: {(cve_df['confidentialityImpact']=='HIGH').sum()}/{n_cves} CVEs
  • Availability  HIGH: {(cve_df['availabilityImpact']=='HIGH').sum()}/{n_cves} CVEs
  • Integrity     HIGH: {(cve_df['integrityImpact']=='HIGH').sum()}/{n_cves} CVEs

TACTIC WITH MOST PATHS:  Defense Evasion (14 paths, 2 CWEs)
TACTIC WITH FEWEST PATHS: Initial Access (1 path, 1 CWE)
MOST COMPLEX CVE: CVE-2014-5432 spans 7 tactics
DOMINANT CWE: Use of Hard-coded Credentials (CWE-798) — 5 CVEs (50%)

ATTACK TREE STRUCTURAL OBSERVATION:
  The framework produced {len(paths_df)} distinct leaf-level attack paths
  distributed across {n_tactics} tactic-level trees. Defense Evasion and
  Privilege Escalation produced the deepest trees (14 and 11 paths), while
  Initial Access produced the simplest (1 path, 1 CWE). This asymmetry
  reflects that the hard-coded credential flaw provides a narrow but direct
  entry vector, whereas once inside, the attacker has a broad set of
  post-exploitation options enabled by the permission misconfiguration in
  CVE-2020-12041 (CWE-732).

COMPARISON POINT (vs. manual / NVD-only approaches):
  A flat NVD lookup yields 10 CVEs with scores and CWE IDs. The CVE2AT
  framework enriches each entry with technique-level (ATT&CK) and
  pattern-level (CAPEC) context, producing {len(paths_df)} structured paths
  that expose the multi-stage nature of the attack surface — an insight
  not available from score-based triage alone.

────────────────────────────────────────────────────────────────────
CONCLUSIONS & FUTURE WORK  →  key factual anchor sentences
────────────────────────────────────────────────────────────────────

  • The case study demonstrates that {n_cves} CVEs, when enriched through
    the CWE–ATT&CK–CAPEC mapping pipeline, yield {len(paths_df)} distinct
    attack paths and {n_tactics} tactic-level attack trees, providing a
    richer basis for threat modelling than score-based ranking alone.

  • The dominance of CWE-798 across 50\\% of CVEs and five tactics (Initial
    Access, Credential Access, Persistence, Privilege Escalation, Defense
    Evasion) illustrates that a single unresolved design flaw can propagate
    across the entire ATT&CK kill chain.

  • All vulnerabilities share attack complexity LOW and require no privileges
    or user interaction, indicating that the identified attack paths are
    immediately exploitable without preconditions — a critical finding for
    clinical risk assessment.

  FUTURE WORK DIRECTIONS:
    1. Extend the framework to multi-device systems (e.g., pump + gateway +
       EHR) where lateral movement paths cross device boundaries.
    2. Automate CAPEC-to-mitigation mapping using CWE Potential Mitigations
       to generate remediation recommendations alongside the attack tree.
    3. Evaluate scalability on larger datasets (hundreds of CVEs) and measure
       path deduplication efficiency across product families.
    4. Integrate CVSS temporal and environmental metrics to weight tree paths
       by deployment-context risk, enabling prioritised remediation ordering.
"""
print(insights)

print(f"\n{'═'*70}")
print(f"  All plots saved to: {os.path.abspath(PLOT_DIR)}/")
print(f"{'═'*70}")

In [1]:
"""
generate_figures_tables.py
--------------------------
Generates all publication-quality figures (PDF + PNG) and LaTeX table code
for the Result Analysis section of the CVE2AT paper.

Outputs written to  ./figures/
    fig1_cvss_scores.pdf/.png
    fig2_tactic_paths.pdf/.png
    fig3_cwe_distribution.pdf/.png
    fig4_cve_tactic_heatmap.pdf/.png
    fig5_capec_risk_matrix.pdf/.png
    fig6_attack_surface_radar.pdf/.png
    tables.tex   ← paste into LaTeX directly

Usage:
    python generate_figures_tables.py

Requirements:
    pip install pandas openpyxl matplotlib numpy
"""

import os
import textwrap
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from matplotlib.patches import FancyBboxPatch
from matplotlib import rcParams

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────────────────
# GLOBAL STYLE  — clean academic look
# ─────────────────────────────────────────────────────────────────────────────
rcParams.update({
    "font.family":        "serif",
    "font.serif":         ["Times New Roman", "DejaVu Serif"],
    "font.size":          11,
    "axes.titlesize":     12,
    "axes.titleweight":   "bold",
    "axes.labelsize":     11,
    "xtick.labelsize":    9,
    "ytick.labelsize":    9,
    "legend.fontsize":    9,
    "figure.dpi":         150,
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.grid":          True,
    "grid.alpha":         0.3,
    "grid.linestyle":     "--",
    "savefig.bbox":       "tight",
    "savefig.dpi":        300,
})

OUT_DIR = "figures"
os.makedirs(OUT_DIR, exist_ok=True)

def save(fig, name):
    for ext in ("pdf", "png"):
        fig.savefig(os.path.join(OUT_DIR, f"{name}.{ext}"))
    plt.close(fig)
    print(f"  ✓  {name}.pdf/.png")


# ─────────────────────────────────────────────────────────────────────────────
# LOAD & PREPARE DATA
# ─────────────────────────────────────────────────────────────────────────────

df = pd.read_excel("device_compromise_dataset.xlsx", index_col=False)
df = df.fillna("")
df["tactics"]   = df["tactics"].astype(str).str.strip()
df["baseScore"] = pd.to_numeric(df["baseScore"],  errors="coerce")
df["exploitabilityScore"] = pd.to_numeric(df["exploitabilityScore"], errors="coerce")

# Merge deprecated/synonym CWE names
CWE_ALIAS = {
    "Use of Hard-coded Password":               "Use of Hard-coded Credentials",
    "DEPRECATED: Authentication Bypass Issues": "Improper Authentication",
}
df["cwe_clean"] = df["cwe_Name"].map(lambda x: CWE_ALIAS.get(x, x))

# Canonical tactic order (ATT&CK kill-chain sequence)
TACTIC_ORDER = [
    "Initial Access", "Credential Access", "Defense Evasion",
    "Privilege Escalation", "Persistence",
    "Discovery", "Lateral Movement", "Collection",
]

# Deduplicated views
cve_df   = df[df["CVE ID"] != ""].drop_duplicates("CVE ID").set_index("CVE ID")
tac_cve  = df[df["tactics"] != ""].drop_duplicates(["CVE ID", "tactics"])
cwe_cve  = df[df["cwe_Name"] != ""].drop_duplicates(["CVE ID", "cwe_clean"])
paths_df = (
    df[df["tactics"] != ""]
    .drop_duplicates(["tactics", "cwe_Name", "technique_name",
                      "subtechnique_name", "capec_Name"])
)
tech_cve = df[df["technique_name"] != ""].drop_duplicates(
    ["CVE ID", "technique_name", "tactics"]
)

# Severity colours (consistent across all figures)
SEV_COLOR = {
    "CRITICAL": "#c0392b",
    "MEDIUM":   "#2980b9",
    "LOW":      "#27ae60",
}
SEV_ORDER = ["CRITICAL", "MEDIUM", "LOW"]

# ─────────────────────────────────────────────────────────────────────────────
# FIGURE 1  — CVSS Base Score per CVE  (horizontal bar, coloured by severity)
# ─────────────────────────────────────────────────────────────────────────────

print("\nGenerating Figure 1 — CVSS scores …")

fig, ax = plt.subplots(figsize=(8, 4.2))

# sort by score descending
cve_sorted = cve_df.sort_values("baseScore", ascending=True)
colors     = [SEV_COLOR.get(s, "#888") for s in cve_sorted["baseSeverity"]]

bars = ax.barh(
    cve_sorted.index, cve_sorted["baseScore"],
    color=colors, edgecolor="white", linewidth=0.6, height=0.65,
)

# score labels inside bars
for bar, score in zip(bars, cve_sorted["baseScore"]):
    x = bar.get_width()
    ax.text(
        x - 0.25, bar.get_y() + bar.get_height() / 2,
        f"{score:.1f}", va="center", ha="right",
        fontsize=9, color="white", fontweight="bold",
    )

# threshold lines
ax.axvline(9.0, color="#c0392b", linestyle="--", linewidth=1.0, alpha=0.8)
ax.axvline(7.0, color="#e67e22", linestyle="--", linewidth=1.0, alpha=0.8)
ax.text(9.05, -0.6, "Critical (9.0)", fontsize=8, color="#c0392b")
ax.text(7.05, -0.6, "High (7.0)",     fontsize=8, color="#e67e22")

ax.set_xlim(0, 11.5)
ax.set_xlabel("CVSS v3.x Base Score")
ax.set_title("CVSS Base Score per CVE")

# legend
patches = [mpatches.Patch(color=SEV_COLOR[s], label=s.capitalize())
           for s in SEV_ORDER if s in cve_df["baseSeverity"].values]
ax.legend(handles=patches, loc="lower right", framealpha=0.9)
ax.grid(axis="x", alpha=0.3, linestyle="--")
ax.set_axisbelow(True)

save(fig, "fig1_cvss_scores")

# ─────────────────────────────────────────────────────────────────────────────
# FIGURE 2  — Attack paths and CVEs per tactic  (grouped bar)
# ─────────────────────────────────────────────────────────────────────────────

print("Generating Figure 2 — Tactic distribution …")

paths_per_tac = (
    paths_df.groupby("tactics").size()
    .reindex(TACTIC_ORDER).fillna(0).astype(int)
)
cves_per_tac = (
    tac_cve.groupby("tactics")["CVE ID"].nunique()
    .reindex(TACTIC_ORDER).fillna(0).astype(int)
)
cwes_per_tac = (
    paths_df.groupby("tactics")["cwe_Name"].nunique()
    .reindex(TACTIC_ORDER).fillna(0).astype(int)
)

x  = np.arange(len(TACTIC_ORDER))
w  = 0.28

fig, ax = plt.subplots(figsize=(10, 4.5))

b1 = ax.bar(x - w,   paths_per_tac.values, width=w, label="Attack paths",
            color="#2c3e50", edgecolor="white", linewidth=0.5)
b2 = ax.bar(x,       cves_per_tac.values,  width=w, label="CVEs",
            color="#2980b9", edgecolor="white", linewidth=0.5)
b3 = ax.bar(x + w,   cwes_per_tac.values,  width=w, label="CWEs",
            color="#27ae60", edgecolor="white", linewidth=0.5)

for bars in (b1, b2, b3):
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            ax.text(
                bar.get_x() + bar.get_width() / 2, h + 0.15,
                str(int(h)), ha="center", va="bottom", fontsize=8,
            )

short_labels = [t.replace(" ", "\n") for t in TACTIC_ORDER]
ax.set_xticks(x)
ax.set_xticklabels(short_labels, fontsize=8.5)
ax.set_ylabel("Count")
ax.set_title("Attack Paths, CVEs, and CWEs per MITRE ATT\\&CK Tactic")
ax.legend(framealpha=0.9)
ax.set_ylim(0, paths_per_tac.max() + 3)
ax.set_axisbelow(True)

save(fig, "fig2_tactic_paths")

# ─────────────────────────────────────────────────────────────────────────────
# FIGURE 3  — CWE distribution  (horizontal bar, sorted)
# ─────────────────────────────────────────────────────────────────────────────

print("Generating Figure 3 — CWE distribution …")

CWE_SHORT = {
    "Use of Hard-coded Credentials":                      "Hard-coded Credentials\n(CWE-798)",
    "Cleartext Storage of Sensitive Information":          "Cleartext Storage\n(CWE-312)",
    "Cleartext Transmission of Sensitive Information":     "Cleartext Transmission\n(CWE-319)",
    "Improper Authentication":                             "Improper Authentication\n(CWE-287)",
    "Incorrect Permission Assignment for Critical Resource":"Incorrect Permission\n(CWE-732)",
    "Operation on a Resource after Expiration or Release": "Resource Expiration\n(CWE-672)",
}

cwe_counts = (
    cwe_cve.groupby("cwe_clean")["CVE ID"].count()
    .sort_values(ascending=True)
)
labels  = [CWE_SHORT.get(k, k) for k in cwe_counts.index]
palette = ["#2c3e50","#2980b9","#27ae60","#8e44ad","#e67e22","#c0392b"]

fig, ax = plt.subplots(figsize=(8, 4.2))
bars = ax.barh(labels, cwe_counts.values,
               color=palette[:len(cwe_counts)],
               edgecolor="white", linewidth=0.5, height=0.6)
for bar, val in zip(bars, cwe_counts.values):
    ax.text(val + 0.05, bar.get_y() + bar.get_height() / 2,
            str(val), va="center", fontsize=10, fontweight="bold")

ax.set_xlim(0, cwe_counts.max() + 1.2)
ax.set_xlabel("Number of CVEs")
ax.set_title("Weakness Class Distribution (CWE)")
ax.set_axisbelow(True)

save(fig, "fig3_cwe_distribution")

# ─────────────────────────────────────────────────────────────────────────────
# FIGURE 4  — CVE × Tactic coverage heatmap
# ─────────────────────────────────────────────────────────────────────────────

print("Generating Figure 4 — CVE–Tactic heatmap …")

all_cves = sorted(cve_df.index.tolist())
heat = pd.DataFrame(0, index=all_cves, columns=TACTIC_ORDER)
for _, row in tac_cve.iterrows():
    if row["CVE ID"] in heat.index and row["tactics"] in heat.columns:
        heat.loc[row["CVE ID"], row["tactics"]] = 1

fig, ax = plt.subplots(figsize=(11, 4.5))
im = ax.imshow(heat.values, aspect="auto", cmap="Blues", vmin=0, vmax=1.3)

ax.set_xticks(range(len(TACTIC_ORDER)))
ax.set_xticklabels(TACTIC_ORDER, rotation=30, ha="right", fontsize=9)
ax.set_yticks(range(len(all_cves)))
ax.set_yticklabels(all_cves, fontsize=9)

# checkmark in filled cells
for i in range(len(all_cves)):
    for j in range(len(TACTIC_ORDER)):
        if heat.values[i, j]:
            ax.text(j, i, "✓", ha="center", va="center",
                    fontsize=13, color="#2c3e50", fontweight="bold")

# column totals along the top
for j, tac in enumerate(TACTIC_ORDER):
    total = int(heat[tac].sum())
    ax.text(j, -0.7, str(total), ha="center", va="center",
            fontsize=9, color="#c0392b", fontweight="bold")
ax.text(-0.6, -0.7, "n→", ha="right", va="center", fontsize=8, color="#c0392b")

# row totals on the right
for i, cid in enumerate(all_cves):
    total = int(heat.loc[cid].sum())
    ax.text(len(TACTIC_ORDER) - 0.4, i, f"  {total}",
            ha="left", va="center", fontsize=9,
            color="#c0392b", fontweight="bold")
ax.text(len(TACTIC_ORDER) - 0.4, -0.7, "  n",
        ha="left", va="center", fontsize=8, color="#c0392b")

ax.set_title("CVE – Tactic Coverage Matrix  (red numbers = totals)")
ax.grid(False)

save(fig, "fig4_cve_tactic_heatmap")

# ─────────────────────────────────────────────────────────────────────────────
# FIGURE 5  — CAPEC risk matrix  (likelihood vs severity, bubble size = count)
# ─────────────────────────────────────────────────────────────────────────────

print("Generating Figure 5 — CAPEC risk matrix …")

LIKE_MAP = {"Low": 1, "Medium": 2, "High": 3}
SEV_MAP  = {"Low": 1, "Medium": 2, "High": 3, "Very High": 4}

capec_risk = (
    df[(df["capec_Likelihood_Of_Attack"] != "") &
       (df["capec_Typical_Severity"] != "")]
    [["capec_Likelihood_Of_Attack", "capec_Typical_Severity", "capec_Name"]]
    .drop_duplicates()
)
capec_risk["like_n"] = capec_risk["capec_Likelihood_Of_Attack"].map(LIKE_MAP)
capec_risk["sev_n"]  = capec_risk["capec_Typical_Severity"].map(SEV_MAP)

# count per (likelihood, severity) cell
cell_counts = (
    capec_risk.groupby(["like_n", "sev_n"]).size().reset_index(name="count")
)

# risk zone background colours
RISK_COLORS = {
    (1,1):"#d5f5e3", (1,2):"#d5f5e3", (1,3):"#fef9e7", (1,4):"#fdebd0",
    (2,1):"#d5f5e3", (2,2):"#fef9e7", (2,3):"#fdebd0", (2,4):"#fadbd8",
    (3,1):"#fef9e7", (3,2):"#fdebd0", (3,3):"#fadbd8", (3,4):"#fadbd8",
}

fig, ax = plt.subplots(figsize=(6.5, 5.5))

# draw zone rectangles
for (lv, sv), col in RISK_COLORS.items():
    rect = FancyBboxPatch(
        (lv - 0.5, sv - 0.5), 1, 1,
        boxstyle="square,pad=0", facecolor=col,
        edgecolor="#bdc3c7", linewidth=0.5, zorder=0,
    )
    ax.add_patch(rect)

# bubble scatter
for _, row in cell_counts.iterrows():
    ax.scatter(
        row["like_n"], row["sev_n"],
        s=row["count"] * 200,
        color="#2c3e50", alpha=0.75, zorder=3,
        edgecolors="white", linewidths=1.2,
    )
    ax.text(row["like_n"], row["sev_n"], str(int(row["count"])),
            ha="center", va="center", fontsize=10,
            color="white", fontweight="bold", zorder=4)

# axes labels
ax.set_xticks([1, 2, 3])
ax.set_xticklabels(["Low", "Medium", "High"], fontsize=10)
ax.set_yticks([1, 2, 3, 4])
ax.set_yticklabels(["Low", "Medium", "High", "Very High"], fontsize=10)
ax.set_xlabel("CAPEC Likelihood of Attack", fontsize=11)
ax.set_ylabel("CAPEC Typical Severity",     fontsize=11)
ax.set_title("CAPEC Risk Matrix\n(bubble size and label = number of attack patterns)")
ax.set_xlim(0.4, 3.6); ax.set_ylim(0.4, 4.6)
ax.grid(False)

# legend for risk zones
leg_patches = [
    mpatches.Patch(color="#d5f5e3", label="Low risk"),
    mpatches.Patch(color="#fef9e7", label="Medium risk"),
    mpatches.Patch(color="#fdebd0", label="High risk"),
    mpatches.Patch(color="#fadbd8", label="Critical risk"),
]
ax.legend(handles=leg_patches, loc="upper left",
          fontsize=8.5, framealpha=0.9)

save(fig, "fig5_capec_risk_matrix")

# ─────────────────────────────────────────────────────────────────────────────
# FIGURE 6  — Attack surface radar  (per CVE: tactics × CIA × exploitability)
# ─────────────────────────────────────────────────────────────────────────────

print("Generating Figure 6 — Attack surface radar …")

IMP_MAP = {"NONE": 0, "LOW": 1, "PARTIAL": 1, "HIGH": 2}

radar_data = []
for cid, row in cve_df.iterrows():
    n_tac  = tac_cve[tac_cve["CVE ID"] == cid]["tactics"].nunique()
    n_path = paths_df.merge(
        pd.DataFrame({"CVE ID": [cid]}), on=None, how="cross"
    )  # just use len workaround below

all_cve_ids = list(cve_df.index)
tac_per_cve   = tac_cve.groupby("CVE ID")["tactics"].count()
path_per_cve  = (
    df[df["tactics"]!=""]
    .drop_duplicates(["CVE ID","tactics","cwe_Name","technique_name","subtechnique_name","capec_Name"])
    .groupby("CVE ID").size()
)

DIMS = ["#Tactics", "CVSS\nScore", "Confidentiality", "Integrity",
        "Availability", "Exploitability"]
N = len(DIMS)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

def norm(vals, vmin, vmax):
    return [(v - vmin) / (vmax - vmin) for v in vals]

fig, ax = plt.subplots(figsize=(7, 6),
                       subplot_kw=dict(polar=True))
ax.set_facecolor("#f9f9f9")

cmap_radar = plt.get_cmap("tab10")
legend_handles = []

for idx, cid in enumerate(sorted(all_cve_ids)):
    row   = cve_df.loc[cid]
    ntac  = tac_per_cve.get(cid, 1)
    score = float(row["baseScore"]) if pd.notna(row["baseScore"]) else 0
    conf  = IMP_MAP.get(str(row["confidentialityImpact"]).upper(), 0)
    integ = IMP_MAP.get(str(row["integrityImpact"]).upper(), 0)
    avail = IMP_MAP.get(str(row["availabilityImpact"]).upper(), 0)
    expl  = float(row["exploitabilityScore"]) if pd.notna(
        row.get("exploitabilityScore")) else 0

    raw    = [ntac, score, conf, integ, avail, expl]
    maxval = [7,    10,    2,    2,     2,     4]
    normed = [v / m for v, m in zip(raw, maxval)]
    normed += normed[:1]

    color = cmap_radar(idx / len(all_cve_ids))
    ax.plot(angles, normed, color=color, linewidth=1.3, linestyle="solid",
            alpha=0.8)
    ax.fill(angles, normed, color=color, alpha=0.06)
    legend_handles.append(
        mpatches.Patch(color=color, label=cid, alpha=0.8)
    )

ax.set_thetagrids(np.degrees(angles[:-1]), DIMS, fontsize=9)
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(["25%", "50%", "75%", "100%"], fontsize=7)
ax.set_title("Attack Surface Profile per CVE\n(normalised dimensions)",
             pad=18, fontsize=12, fontweight="bold")
ax.legend(handles=legend_handles, loc="upper right",
          bbox_to_anchor=(1.42, 1.15), fontsize=7.5, framealpha=0.9)

save(fig, "fig6_attack_surface_radar")

# ─────────────────────────────────────────────────────────────────────────────
# LATEX TABLES  — written to figures/tables.tex
# ─────────────────────────────────────────────────────────────────────────────

print("Generating LaTeX tables …")

tex_lines = []

def tex(s=""):
    tex_lines.append(s)

# ── Table 1: Tactic distribution ────────────────────────────────────────────
tex("% ════════════════════════════════════════════════════════════════════")
tex("% TABLE 1 — Attack path distribution across MITRE ATT&CK tactics")
tex("% Required packages: booktabs, array")
tex("% ════════════════════════════════════════════════════════════════════")
tex(r"\begin{table}[ht]")
tex(r"\centering")
tex(r"\caption{Distribution of unique attack paths across the eight MITRE")
tex(r"ATT\&CK tactics identified in the dataset.}")
tex(r"\label{tab:tactic_distribution}")
tex(r"\renewcommand{\arraystretch}{1.25}")
tex(r"\begin{tabular}{@{} l r r r r @{}}")
tex(r"\toprule")
tex(r"\textbf{Tactic} & \textbf{CVEs} & \textbf{Paths}"
    r" & \textbf{CWEs} & \textbf{Techniques} \\")
tex(r"\midrule")

for tac in TACTIC_ORDER:
    tdf  = paths_df[paths_df["tactics"] == tac]
    npth = len(tdf)
    ncve = int(cves_per_tac.get(tac, 0))
    ncwe = tdf["cwe_Name"].nunique()
    ntech= tdf["technique_name"].nunique()
    if npth > 0:
        tex(f"{tac} & {ncve} & {npth} & {ncwe} & {ntech} \\\\")

tex(r"\midrule")
tex(f"\\textbf{{Total}} & "
    f"\\textbf{{{len(cve_df)}}} & "
    f"\\textbf{{{len(paths_df)}}} & "
    f"\\textbf{{{paths_df['cwe_Name'].nunique()}}} & "
    f"\\textbf{{{paths_df['technique_name'].nunique()}}} \\\\")
tex(r"\bottomrule")
tex(r"\end{tabular}")
tex(r"\end{table}")
tex()

# ── Table 2: CWE summary ────────────────────────────────────────────────────
tex("% ════════════════════════════════════════════════════════════════════")
tex("% TABLE 2 — Weakness class distribution")
tex("% Required packages: booktabs, tabularx")
tex("% ════════════════════════════════════════════════════════════════════")
tex(r"\begin{table}[ht]")
tex(r"\centering")
tex(r"\caption{Weakness classes identified in the dataset and their")
tex(r"occurrence across the ten CVEs.}")
tex(r"\label{tab:cwe_distribution}")
tex(r"\renewcommand{\arraystretch}{1.25}")
tex(r"\begin{tabularx}{\columnwidth}{@{} X r c @{}}")
tex(r"\toprule")
tex(r"\textbf{Weakness class} & \textbf{CVEs} & \textbf{Tactics} \\")
tex(r"\midrule")

for cwe, cnt in cwe_counts.sort_values(ascending=False).items():
    # number of tactics this CWE appears in
    n_tac = (
        paths_df[paths_df["cwe_clean"] == cwe]["tactics"].nunique()
        if "cwe_clean" in paths_df.columns
        else "—"
    )
    safe = cwe.replace("&", r"\&")
    tex(f"{safe} & {cnt} & {n_tac} \\\\")

tex(r"\bottomrule")
tex(r"\end{tabularx}")
tex(r"\end{table}")
tex()

# ── Table 3: per-CVE CVSS summary ───────────────────────────────────────────
tex("% ════════════════════════════════════════════════════════════════════")
tex("% TABLE 3 — Per-CVE CVSS metrics and tactic breadth")
tex("% Required packages: booktabs, tabularx, multirow")
tex("% ════════════════════════════════════════════════════════════════════")
tex(r"\begin{table*}[ht]")
tex(r"\centering")
tex(r"\caption{CVSS metrics and tactic breadth for each CVE in the dataset.")
tex(r"AV = Attack Vector (N = Network, P = Physical).")
tex(r"C/I/A = Confidentiality, Integrity, Availability Impact (H = High,")
tex(r"L = Low, N = None).}")
tex(r"\label{tab:cve_cvss}")
tex(r"\renewcommand{\arraystretch}{1.25}")
tex(r"\begin{tabularx}{\textwidth}{@{} l c c c c c c c r @{}}")
tex(r"\toprule")
tex(r"\textbf{CVE} & \textbf{Score} & \textbf{Severity}"
    r" & \textbf{AV} & \textbf{C} & \textbf{I} & \textbf{A}"
    r" & \textbf{Expl.} & \textbf{Tactics} \\")
tex(r"\midrule")

sev_cmd = {"CRITICAL": r"\textbf{Critical}", "MEDIUM": "Medium", "LOW": "Low"}
imp_short = {"HIGH": "H", "LOW": "L", "NONE": "N", "PARTIAL": "P"}
av_short  = {"NETWORK": "N", "PHYSICAL": "P"}

for cid in sorted(all_cve_ids):
    row  = cve_df.loc[cid]
    ntac = int(tac_per_cve.get(cid, 0))
    sev  = sev_cmd.get(str(row["baseSeverity"]), str(row["baseSeverity"]))
    av   = av_short.get(str(row["attackVector"]), str(row["attackVector"]))
    c    = imp_short.get(str(row["confidentialityImpact"]).upper(), "—")
    i    = imp_short.get(str(row["integrityImpact"]).upper(),       "—")
    a    = imp_short.get(str(row["availabilityImpact"]).upper(),    "—")
    sc   = f"{float(row['baseScore']):.1f}" if pd.notna(row["baseScore"]) else "—"
    ex   = f"{float(row['exploitabilityScore']):.1f}" if pd.notna(
        row.get("exploitabilityScore")) else "—"
    tex(f"\\texttt{{{cid}}} & {sc} & {sev} & {av}"
        f" & {c} & {i} & {a} & {ex} & {ntac} \\\\")

tex(r"\bottomrule")
tex(r"\end{tabularx}")
tex(r"\end{table*}")
tex()

# ── Table 4: Technique distribution ─────────────────────────────────────────
tex("% ════════════════════════════════════════════════════════════════════")
tex("% TABLE 4 — Top techniques and their tactic coverage")
tex("% ════════════════════════════════════════════════════════════════════")
tex(r"\begin{table}[ht]")
tex(r"\centering")
tex(r"\caption{MITRE ATT\&CK techniques identified in the dataset,")
tex(r"ordered by occurrence.}")
tex(r"\label{tab:technique_distribution}")
tex(r"\renewcommand{\arraystretch}{1.25}")
tex(r"\begin{tabularx}{\columnwidth}{@{} X r r @{}}")
tex(r"\toprule")
tex(r"\textbf{Technique} & \textbf{Occurrences} & \textbf{Tactics} \\")
tex(r"\midrule")

tech_counts = tech_cve["technique_name"].value_counts()
for tech, cnt in tech_counts.items():
    n_tac = tech_cve[tech_cve["technique_name"] == tech]["tactics"].nunique()
    safe  = tech.replace("&", r"\&")
    tex(f"{safe} & {cnt} & {n_tac} \\\\")

tex(r"\bottomrule")
tex(r"\end{tabularx}")
tex(r"\end{table}")

# write file
tables_path = os.path.join(OUT_DIR, "tables.tex")
with open(tables_path, "w", encoding="utf-8") as f:
    f.write("\n".join(tex_lines))
print(f"  ✓  tables.tex  ({len(tex_lines)} lines)")

# ─────────────────────────────────────────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────────────────────────────────────────

print(f"""
{'─'*60}
All outputs saved to:  {os.path.abspath(OUT_DIR)}/

  Figures (PDF + PNG):
    fig1_cvss_scores          — CVSS score per CVE (horizontal bar)
    fig2_tactic_paths         — Paths / CVEs / CWEs per tactic (grouped bar)
    fig3_cwe_distribution     — Weakness class distribution (horizontal bar)
    fig4_cve_tactic_heatmap   — CVE × Tactic coverage matrix (heatmap)
    fig5_capec_risk_matrix    — CAPEC likelihood × severity (bubble matrix)
    fig6_attack_surface_radar — Per-CVE attack surface profile (radar)

  LaTeX tables (paste into your .tex file):
    tables.tex
      Table 1  tab:tactic_distribution  — tactic × paths/CVEs/CWEs/techs
      Table 2  tab:cwe_distribution     — CWEs with CVE count and tactic breadth
      Table 3  tab:cve_cvss             — per-CVE CVSS metrics + tactic breadth
      Table 4  tab:technique_distribution — technique frequency

  LaTeX usage example:
    \\input{{figures/tables}}
    \\includegraphics[width=\\columnwidth]{{figures/fig1_cvss_scores}}
{'─'*60}
""")


Generating Figure 1 — CVSS scores …
  ✓  fig1_cvss_scores.pdf/.png
Generating Figure 2 — Tactic distribution …
  ✓  fig2_tactic_paths.pdf/.png
Generating Figure 3 — CWE distribution …
  ✓  fig3_cwe_distribution.pdf/.png
Generating Figure 4 — CVE–Tactic heatmap …
  ✓  fig4_cve_tactic_heatmap.pdf/.png
Generating Figure 5 — CAPEC risk matrix …
  ✓  fig5_capec_risk_matrix.pdf/.png
Generating Figure 6 — Attack surface radar …
  ✓  fig6_attack_surface_radar.pdf/.png
Generating LaTeX tables …
  ✓  tables.tex  (114 lines)

────────────────────────────────────────────────────────────
All outputs saved to:  /home/user/Documents/Bits goa/Phd Projects/Working_Directory/INFUSION_PUMP_FROM_adversary_Perspective/Oraganized-Threat-Intelligence-Code/3.Visualizations/figures/

  Figures (PDF + PNG):
    fig1_cvss_scores          — CVSS score per CVE (horizontal bar)
    fig2_tactic_paths         — Paths / CVEs / CWEs per tactic (grouped bar)
    fig3_cwe_distribution     — Weakness class distribution (

In [2]:
"""
generate_figures_consolidated.py
---------------------------------
Generates 3 consolidated, publication-quality composite figures (PDF + PNG)
for the CVE2AT Result Analysis section.

Figure layout:
  fig1_vulnerability_profile   — CVSS bars (left) + CWE distribution (right)
  fig2_attack_structure        — Tactic grouped bar (top) + CVE–Tactic heatmap (bottom)
  fig3_threat_landscape        — CAPEC risk matrix (left) + Attack surface radar (right)

Usage:
    python generate_figures_consolidated.py

Requirements:
    pip install pandas openpyxl matplotlib numpy
"""

import os, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
from matplotlib.patches import FancyBboxPatch
from matplotlib import rcParams

warnings.filterwarnings("ignore")

# ─── GLOBAL ACADEMIC STYLE ────────────────────────────────────────────────────
rcParams.update({
    "font.family":        "serif",
    "font.serif":         ["Times New Roman", "DejaVu Serif"],
    "font.size":          10,
    "axes.titlesize":     10,
    "axes.titleweight":   "bold",
    "axes.labelsize":     9.5,
    "xtick.labelsize":    8.5,
    "ytick.labelsize":    8.5,
    "legend.fontsize":    8,
    "figure.dpi":         150,
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.grid":          True,
    "grid.alpha":         0.25,
    "grid.linestyle":     "--",
    "savefig.bbox":       "tight",
    "savefig.dpi":        300,
    "axes.linewidth":     0.7,
})

OUT_DIR = "figures"
os.makedirs(OUT_DIR, exist_ok=True)

def save(fig, name):
    for ext in ("pdf", "png"):
        fig.savefig(os.path.join(OUT_DIR, f"{name}.{ext}"),
                    bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  ✓  {name}.pdf/.png")


# ─── LOAD & PREPARE DATA ─────────────────────────────────────────────────────

df = pd.read_excel("device_compromise_dataset.xlsx", index_col=False)
df = df.fillna("")
df["tactics"]   = df["tactics"].astype(str).str.strip()
df["baseScore"] = pd.to_numeric(df["baseScore"],  errors="coerce")
df["exploitabilityScore"] = pd.to_numeric(df["exploitabilityScore"], errors="coerce")

CWE_ALIAS = {
    "Use of Hard-coded Password":               "Use of Hard-coded Credentials",
    "DEPRECATED: Authentication Bypass Issues": "Improper Authentication",
}
df["cwe_clean"] = df["cwe_Name"].map(lambda x: CWE_ALIAS.get(x, x))

TACTIC_ORDER = [
    "Initial Access", "Credential Access", "Defense Evasion",
    "Privilege Escalation", "Persistence",
    "Discovery", "Lateral Movement", "Collection",
]

cve_df   = df[df["CVE ID"] != ""].drop_duplicates("CVE ID").set_index("CVE ID")
tac_cve  = df[df["tactics"] != ""].drop_duplicates(["CVE ID", "tactics"])
cwe_cve  = df[df["cwe_Name"] != ""].drop_duplicates(["CVE ID", "cwe_clean"])
paths_df = df[df["tactics"] != ""].drop_duplicates(
    ["tactics", "cwe_Name", "technique_name", "subtechnique_name", "capec_Name"])
tech_cve = df[df["technique_name"] != ""].drop_duplicates(
    ["CVE ID", "technique_name", "tactics"])

# Derived counts
paths_per_tac = (paths_df.groupby("tactics").size()
    .reindex(TACTIC_ORDER).fillna(0).astype(int))
cves_per_tac  = (tac_cve.groupby("tactics")["CVE ID"].nunique()
    .reindex(TACTIC_ORDER).fillna(0).astype(int))
cwes_per_tac  = (paths_df.groupby("tactics")["cwe_Name"].nunique()
    .reindex(TACTIC_ORDER).fillna(0).astype(int))
tac_per_cve   = tac_cve.groupby("CVE ID")["tactics"].count()

CWE_SHORT = {
    "Use of Hard-coded Credentials":                      "Hard-coded Credentials\n(CWE-798)",
    "Cleartext Storage of Sensitive Information":          "Cleartext Storage\n(CWE-312)",
    "Cleartext Transmission of Sensitive Information":     "Cleartext Transmission\n(CWE-319)",
    "Improper Authentication":                             "Improper Authentication\n(CWE-287)",
    "Incorrect Permission Assignment for Critical Resource":"Incorrect Permission\n(CWE-732)",
    "Operation on a Resource after Expiration or Release": "Resource Expiration\n(CWE-672)",
}
cwe_counts = cwe_cve.groupby("cwe_clean")["CVE ID"].count().sort_values(ascending=True)

SEV_COLOR = {"CRITICAL": "#c0392b", "MEDIUM": "#2980b9", "LOW": "#27ae60"}

all_cves = sorted(cve_df.index.tolist())

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 1 — Vulnerability Profile
#   Left:  CVSS base score per CVE (horizontal bar, coloured by severity)
#   Right: CWE weakness class distribution (horizontal bar)
# ═══════════════════════════════════════════════════════════════════════════════

print("\nGenerating Figure 1 — Vulnerability Profile …")

fig = plt.figure(figsize=(13, 5.5))
gs  = gridspec.GridSpec(1, 2, figure=fig, wspace=0.38,
                        width_ratios=[1.1, 0.9])

# ── LEFT: CVSS scores ────────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0])

cve_sorted = cve_df.sort_values("baseScore", ascending=True)
colors_bar  = [SEV_COLOR.get(s, "#888") for s in cve_sorted["baseSeverity"]]

bars = ax1.barh(cve_sorted.index, cve_sorted["baseScore"],
                color=colors_bar, edgecolor="white", linewidth=0.5, height=0.65)
for bar, score in zip(bars, cve_sorted["baseScore"]):
    ax1.text(bar.get_width() - 0.22, bar.get_y() + bar.get_height() / 2,
             f"{score:.1f}", va="center", ha="right",
             fontsize=8.5, color="white", fontweight="bold")

ax1.axvline(9.0, color="#c0392b", linestyle="--", lw=1.0, alpha=0.85,
            label="Critical (9.0)")
ax1.axvline(7.0, color="#e67e22", linestyle="--", lw=1.0, alpha=0.85,
            label="High (7.0)")

# severity patches legend
patches = [mpatches.Patch(color=SEV_COLOR[s], label=s.capitalize())
           for s in ["CRITICAL","MEDIUM","LOW"] if s in cve_df["baseSeverity"].values]
# threshold lines
patches += [
    mpatches.Patch(color="#c0392b", alpha=0, label=""),
    plt.Line2D([0],[0], color="#c0392b", ls="--", lw=1, label="Critical (9.0)"),
    plt.Line2D([0],[0], color="#e67e22", ls="--", lw=1, label="High (7.0)"),
]
ax1.legend(handles=patches, fontsize=7.5, loc="lower right",
           framealpha=0.85, handlelength=1.4)

ax1.set_xlim(0, 11.8)
ax1.set_xlabel("CVSS v3.x Base Score")
ax1.set_title("(a) CVSS Base Score per CVE", loc="left", pad=6)
ax1.grid(axis="x"); ax1.set_axisbelow(True)

# ── RIGHT: CWE distribution ──────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[1])

palette = ["#5d6d7e","#2980b9","#27ae60","#8e44ad","#e67e22","#c0392b"]
labels_cwe = [CWE_SHORT.get(k, k) for k in cwe_counts.index]

bars2 = ax2.barh(labels_cwe, cwe_counts.values,
                 color=palette[:len(cwe_counts)],
                 edgecolor="white", linewidth=0.5, height=0.6)
for bar, val in zip(bars2, cwe_counts.values):
    ax2.text(val + 0.06, bar.get_y() + bar.get_height() / 2,
             str(int(val)), va="center", fontsize=9, fontweight="bold",
             color="#2c3e50")

ax2.set_xlim(0, cwe_counts.max() + 1.5)
ax2.set_xlabel("Number of CVEs")
ax2.set_title("(b) Weakness Class (CWE) Distribution", loc="left", pad=6)
ax2.grid(axis="x"); ax2.set_axisbelow(True)

fig.suptitle("Figure 1 — Vulnerability Profile of the CVE Dataset",
             fontsize=11, fontweight="bold", y=1.01)

save(fig, "fig1_vulnerability_profile")


# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 2 — Attack Structure
#   Top:    Attack paths, CVEs, CWEs per tactic (grouped bar)
#   Bottom: CVE × Tactic coverage heatmap
# ═══════════════════════════════════════════════════════════════════════════════

print("Generating Figure 2 — Attack Structure …")

fig = plt.figure(figsize=(13, 9.5))
gs  = gridspec.GridSpec(2, 1, figure=fig, hspace=0.48,
                        height_ratios=[1, 0.95])

# ── TOP: grouped bar per tactic ──────────────────────────────────────────────
ax_top = fig.add_subplot(gs[0])

x = np.arange(len(TACTIC_ORDER))
w = 0.26

b1 = ax_top.bar(x - w,   paths_per_tac.values, width=w, label="Attack paths",
                color="#2c3e50", edgecolor="white", lw=0.5)
b2 = ax_top.bar(x,       cves_per_tac.values,  width=w, label="CVEs",
                color="#2980b9", edgecolor="white", lw=0.5)
b3 = ax_top.bar(x + w,   cwes_per_tac.values,  width=w, label="CWEs",
                color="#27ae60", edgecolor="white", lw=0.5)

for bars_ in (b1, b2, b3):
    for bar in bars_:
        h = bar.get_height()
        if h > 0:
            ax_top.text(bar.get_x() + bar.get_width() / 2, h + 0.15,
                        str(int(h)), ha="center", va="bottom", fontsize=7.5)

short_labels = [t.replace(" ", "\n") for t in TACTIC_ORDER]
ax_top.set_xticks(x)
ax_top.set_xticklabels(short_labels, fontsize=8)
ax_top.set_ylabel("Count")
ax_top.set_title("(a) Attack Paths, CVEs, and CWEs per MITRE ATT\u0026CK Tactic",
                 loc="left", pad=6)
ax_top.legend(framealpha=0.9, loc="upper right")
ax_top.set_ylim(0, paths_per_tac.max() + 3)
ax_top.set_axisbelow(True)

# ── BOTTOM: CVE × Tactic heatmap ─────────────────────────────────────────────
ax_bot = fig.add_subplot(gs[1])

heat = pd.DataFrame(0, index=all_cves, columns=TACTIC_ORDER)
for _, row in tac_cve.iterrows():
    if row["CVE ID"] in heat.index and row["tactics"] in heat.columns:
        heat.loc[row["CVE ID"], row["tactics"]] = 1

ax_bot.imshow(heat.values, aspect="auto", cmap="Blues", vmin=0, vmax=1.4)

ax_bot.set_xticks(range(len(TACTIC_ORDER)))
ax_bot.set_xticklabels(TACTIC_ORDER, rotation=28, ha="right", fontsize=8.5)
ax_bot.set_yticks(range(len(all_cves)))
ax_bot.set_yticklabels(all_cves, fontsize=8)

for i in range(len(all_cves)):
    for j in range(len(TACTIC_ORDER)):
        if heat.values[i, j]:
            ax_bot.text(j, i, "✓", ha="center", va="center",
                        fontsize=12, color="#2c3e50", fontweight="bold")

# column totals
for j, tac in enumerate(TACTIC_ORDER):
    ax_bot.text(j, -0.75, str(int(heat[tac].sum())),
                ha="center", va="center", fontsize=8,
                color="#c0392b", fontweight="bold")
ax_bot.text(-0.65, -0.75, "Σ", ha="right", va="center",
            fontsize=8, color="#c0392b", fontweight="bold")

# row totals
for i, cid in enumerate(all_cves):
    ax_bot.text(len(TACTIC_ORDER) - 0.35, i, f"  {int(heat.loc[cid].sum())}",
                ha="left", va="center", fontsize=8,
                color="#c0392b", fontweight="bold")
ax_bot.text(len(TACTIC_ORDER) - 0.35, -0.75, "  Σ",
            ha="left", va="center", fontsize=8,
            color="#c0392b", fontweight="bold")

ax_bot.set_title("(b) CVE–Tactic Coverage Matrix  (Σ = row/column totals in red)",
                 loc="left", pad=6)
ax_bot.grid(False)
for spine in ax_bot.spines.values():
    spine.set_visible(False)

fig.suptitle("Figure 2 — Attack Structure: Tactic Distribution and CVE Coverage",
             fontsize=11, fontweight="bold", y=1.005)

save(fig, "fig2_attack_structure")


# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 3 — Threat Landscape
#   Left:  CAPEC risk matrix (likelihood × severity, bubble = count)
#   Right: Per-CVE attack surface radar (normalised)
# ═══════════════════════════════════════════════════════════════════════════════

print("Generating Figure 3 — Threat Landscape …")

LIKE_MAP = {"Low": 1, "Medium": 2, "High": 3}
SEV_MAP  = {"Low": 1, "Medium": 2, "High": 3, "Very High": 4}

capec_risk = (
    df[(df["capec_Likelihood_Of_Attack"] != "") &
       (df["capec_Typical_Severity"] != "")]
    [["capec_Likelihood_Of_Attack", "capec_Typical_Severity", "capec_Name"]]
    .drop_duplicates()
)
capec_risk["like_n"] = capec_risk["capec_Likelihood_Of_Attack"].map(LIKE_MAP)
capec_risk["sev_n"]  = capec_risk["capec_Typical_Severity"].map(SEV_MAP)

cell_counts = (capec_risk.groupby(["like_n", "sev_n"]).size()
               .reset_index(name="count"))

RISK_COLORS = {
    (1,1):"#d5f5e3",(1,2):"#d5f5e3",(1,3):"#fef9e7",(1,4):"#fdebd0",
    (2,1):"#d5f5e3",(2,2):"#fef9e7",(2,3):"#fdebd0",(2,4):"#fadbd8",
    (3,1):"#fef9e7",(3,2):"#fdebd0",(3,3):"#fadbd8",(3,4):"#fadbd8",
}

IMP_MAP = {"NONE": 0, "LOW": 1, "PARTIAL": 1, "HIGH": 2}

fig = plt.figure(figsize=(13, 5.8))
gs  = gridspec.GridSpec(1, 2, figure=fig, wspace=0.35, width_ratios=[1, 1.05])

# ── LEFT: CAPEC risk matrix ───────────────────────────────────────────────────
ax_l = fig.add_subplot(gs[0])

for (lv, sv), col in RISK_COLORS.items():
    rect = FancyBboxPatch((lv-0.5, sv-0.5), 1, 1,
                          boxstyle="square,pad=0",
                          facecolor=col, edgecolor="#bdc3c7",
                          linewidth=0.6, zorder=0)
    ax_l.add_patch(rect)

for _, row in cell_counts.iterrows():
    ax_l.scatter(row["like_n"], row["sev_n"],
                 s=row["count"] * 220,
                 color="#2c3e50", alpha=0.78, zorder=3,
                 edgecolors="white", linewidths=1.2)
    ax_l.text(row["like_n"], row["sev_n"], str(int(row["count"])),
              ha="center", va="center", fontsize=10,
              color="white", fontweight="bold", zorder=4)

ax_l.set_xticks([1, 2, 3])
ax_l.set_xticklabels(["Low", "Medium", "High"])
ax_l.set_yticks([1, 2, 3, 4])
ax_l.set_yticklabels(["Low", "Medium", "High", "Very High"])
ax_l.set_xlabel("CAPEC Likelihood of Attack")
ax_l.set_ylabel("CAPEC Typical Severity")
ax_l.set_title("(a) CAPEC Risk Matrix\n(bubble label = # attack patterns)", loc="left", pad=4)
ax_l.set_xlim(0.4, 3.6); ax_l.set_ylim(0.4, 4.6)
ax_l.grid(False)
for spine in ax_l.spines.values():
    spine.set_visible(False)

leg_patches = [
    mpatches.Patch(color="#d5f5e3", label="Low risk"),
    mpatches.Patch(color="#fef9e7", label="Medium risk"),
    mpatches.Patch(color="#fdebd0", label="High risk"),
    mpatches.Patch(color="#fadbd8", label="Critical risk"),
]
ax_l.legend(handles=leg_patches, loc="upper left", fontsize=7.5, framealpha=0.9)

# ── RIGHT: Attack surface radar ───────────────────────────────────────────────
ax_r = fig.add_subplot(gs[1], polar=True)
ax_r.set_facecolor("#f9f9f9")

DIMS   = ["Tactics", "CVSS\nScore", "Confidentiality", "Integrity",
          "Availability", "Exploitability"]
N      = len(DIMS)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]

cmap_radar = plt.get_cmap("tab10")
legend_handles = []

for idx, cid in enumerate(sorted(all_cves)):
    row   = cve_df.loc[cid]
    ntac  = int(tac_per_cve.get(cid, 1))
    score = float(row["baseScore"]) if pd.notna(row["baseScore"]) else 0
    conf  = IMP_MAP.get(str(row["confidentialityImpact"]).upper(), 0)
    integ = IMP_MAP.get(str(row["integrityImpact"]).upper(), 0)
    avail = IMP_MAP.get(str(row["availabilityImpact"]).upper(), 0)
    expl  = float(row["exploitabilityScore"]) if pd.notna(row.get("exploitabilityScore")) else 0

    raw    = [ntac, score, conf, integ, avail, expl]
    maxval = [7,    10,    2,    2,     2,     4]
    normed = [v/m for v, m in zip(raw, maxval)] + [0]*1
    normed[-1] = normed[0]

    color = cmap_radar(idx / max(len(all_cves)-1, 1))
    ax_r.plot(angles, normed, color=color, lw=1.3, alpha=0.8)
    ax_r.fill(angles, normed, color=color, alpha=0.07)
    legend_handles.append(mpatches.Patch(color=color, label=cid, alpha=0.85))

ax_r.set_thetagrids(np.degrees(angles[:-1]), DIMS, fontsize=8.5)
ax_r.set_ylim(0, 1)
ax_r.set_yticks([0.25, 0.5, 0.75, 1.0])
ax_r.set_yticklabels(["25%","50%","75%","100%"], fontsize=6.5)
ax_r.set_title("(b) Attack Surface Profile per CVE\n(normalised dimensions)",
               loc="center", pad=14, fontsize=9.5, fontweight="bold")
ax_r.legend(handles=legend_handles, loc="upper right",
            bbox_to_anchor=(1.55, 1.18), fontsize=6.5, framealpha=0.9,
            ncol=1, handlelength=1.2)

fig.suptitle("Figure 3 — Threat Landscape: Attack Pattern Risk and Surface Profiles",
             fontsize=11, fontweight="bold", y=1.02)

save(fig, "fig3_threat_landscape")

print(f"""
{'─'*60}
Done. Outputs in: {os.path.abspath(OUT_DIR)}/

  fig1_vulnerability_profile   — CVSS scores + CWE distribution
  fig2_attack_structure        — Tactic bar chart + CVE-tactic heatmap
  fig3_threat_landscape        — CAPEC risk matrix + radar chart
{'─'*60}
""")


Generating Figure 1 — Vulnerability Profile …
  ✓  fig1_vulnerability_profile.pdf/.png
Generating Figure 2 — Attack Structure …
  ✓  fig2_attack_structure.pdf/.png
Generating Figure 3 — Threat Landscape …
  ✓  fig3_threat_landscape.pdf/.png

────────────────────────────────────────────────────────────
Done. Outputs in: /home/pranam/Documents/Bits-goa/Phd/Working_Directory/INFUSION_PUMP_FROM_adversary_Perspective/Oraganized-Threat-Intelligence-Code/3.Visualizations/figures/

  fig1_vulnerability_profile   — CVSS scores + CWE distribution
  fig2_attack_structure        — Tactic bar chart + CVE-tactic heatmap
  fig3_threat_landscape        — CAPEC risk matrix + radar chart
────────────────────────────────────────────────────────────



In [12]:
"""
generate_radar_final.py
------------------------
Generates the attack surface profile radar chart (Fig 8) for the CVE2AT paper.
Reads directly from device_compromise_dataset.xlsx.

TUNING PARAMETERS (easy to adjust):
    DATASET   — path to your Excel file
    PAD       — how far labels sit from the plot edge (increase to push further out)
    FIGSIZE   — (width, height) of the figure in inches
    LABEL_FS  — font size of the axis labels
    TICK_FS   — font size of the % tick labels
    LEG_FS    — font size of the CVE legend
    TITLE_PAD — space between title and plot

Usage:
    python generate_radar_final.py

Output:
    fig8_radar.pdf  — vector, include in LaTeX with \\includegraphics
    fig8_radar.png  — 300 dpi raster, for preview

Requirements:
    pip install pandas openpyxl matplotlib numpy
"""

import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib import rcParams
import os

warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════════
#  TUNING — change these values to adjust the figure
# ══════════════════════════════════════════════════════════════════

DATASET   = "device_compromise_dataset.xlsx"  # path to your Excel file

PAD       = 1.17    # label distance from plot edge
                    # 1.0 = touching the edge, 1.3 = further out
                    # increase in steps of 0.02 if labels still overlap

FIGSIZE   = (7, 6)  # (width, height) in inches

LABEL_FS  = 9       # font size of axis dimension labels
TICK_FS   = 7       # font size of 25%/50%/75%/100% ring labels
LEG_FS    = 10     # font size of CVE legend entries
TITLE_PAD = 28      # space between title text and plot centre
os.makedirs("figure", exist_ok=True)
OUTPUT_PDF = "figure/fig8_radar.pdf"
OUTPUT_PNG = "figure/fig8_radar.png"

# ══════════════════════════════════════════════════════════════════
#  STYLE — academic serif font, high DPI output
# ══════════════════════════════════════════════════════════════════

rcParams.update({
    "font.family":      "serif",
    "font.serif":       ["Times New Roman", "DejaVu Serif"],
    "font.size":        10,
    "axes.titlesize":   10,
    "axes.titleweight": "bold",
    "figure.dpi":       150,
    "savefig.bbox":     "tight",
    "savefig.dpi":      300,
})

# ══════════════════════════════════════════════════════════════════
#  DATA LOADING
# ══════════════════════════════════════════════════════════════════

df = pd.read_excel(DATASET, index_col=False)
df = df.fillna("")
df["tactics"]             = df["tactics"].astype(str).str.strip()
df["baseScore"]           = pd.to_numeric(df["baseScore"],           errors="coerce")
df["exploitabilityScore"] = pd.to_numeric(df["exploitabilityScore"], errors="coerce")

# One row per CVE
cve_df = df[df["CVE ID"] != ""].drop_duplicates("CVE ID").set_index("CVE ID")

# Count how many tactics each CVE maps to
tac_cve     = df[df["tactics"] != ""].drop_duplicates(["CVE ID", "tactics"])
tac_per_cve = tac_cve.groupby("CVE ID")["tactics"].count()
all_cves    = sorted(cve_df.index.tolist())

# CIA impact string → numeric
IMP_MAP = {"NONE": 0, "LOW": 1, "PARTIAL": 1, "HIGH": 2}

# ══════════════════════════════════════════════════════════════════
#  RADAR DIMENSIONS
#  Each dimension is normalised by its MAXVAL so all axes are 0–1
# ══════════════════════════════════════════════════════════════════

DIMS   = ["Tactics", "CVSS\nScore", "Confidentiality",
          "Integrity", "Availability", "Exploitability"]
MAXVAL = [7,          10,             2,
          2,           2,              4]

N      = len(DIMS)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]   # close the polygon

# ══════════════════════════════════════════════════════════════════
#  PLOT
# ══════════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=FIGSIZE, subplot_kw=dict(polar=True))
ax.set_facecolor("#f9f9f9")

cmap           = plt.get_cmap("tab10")
legend_handles = []

for idx, cid in enumerate(sorted(all_cves)):
    row   = cve_df.loc[cid]
    ntac  = int(tac_per_cve.get(cid, 1))
    score = float(row["baseScore"])           if pd.notna(row["baseScore"])           else 0.0
    conf  = IMP_MAP.get(str(row["confidentialityImpact"]).upper(), 0)
    integ = IMP_MAP.get(str(row["integrityImpact"]).upper(),       0)
    avail = IMP_MAP.get(str(row["availabilityImpact"]).upper(),    0)
    expl  = float(row["exploitabilityScore"]) if pd.notna(row.get("exploitabilityScore")) else 0.0

    raw    = [ntac, score, conf, integ, avail, expl]
    normed = [v / m for v, m in zip(raw, MAXVAL)]
    normed += normed[:1]   # close polygon

    color = cmap(idx / max(len(all_cves) - 1, 1))
    ax.plot(angles, normed, color=color, lw=1.3, alpha=0.85)
    ax.fill(angles, normed, color=color, alpha=0.07)
    legend_handles.append(mpatches.Patch(color=color, label=cid, alpha=0.85))

# ── Axis labels placed manually at distance PAD ───────────────────
ax.set_thetagrids(np.degrees(angles[:-1]), labels=[""] * N)  # clear defaults

for angle, label in zip(angles[:-1], DIMS):
    deg = np.degrees(angle)
    if deg < 10 or deg > 350:
        ha = "center"
    elif 10 <= deg <= 170:
        ha = "left"
    elif 170 < deg <= 190:
        ha = "center"
    else:
        ha = "right"
    ax.text(angle, PAD, label,
            ha=ha, va="center",
            fontsize=LABEL_FS, fontfamily="serif")

# ── Ring tick labels ──────────────────────────────────────────────
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.50, 0.75, 1.00])
ax.set_yticklabels(["25 %", "50 %", "75 %", "100 %"], fontsize=TICK_FS)
ax.set_rlabel_position(10)   # move % labels to 30-degree position

# ── Title ─────────────────────────────────────────────────────────
ax.set_title(
    "Normalised Attack Surface Profile per CVE",
    pad=TITLE_PAD, fontsize=10, fontweight="bold",
)

# ── Legend ────────────────────────────────────────────────────────
ax.legend(
    handles=legend_handles,
    loc="upper right",
    bbox_to_anchor=(1.48, 1.18),
    fontsize=LEG_FS,
    framealpha=0.9,
    ncol=1,
    handlelength=1.2,
    title="CVE identifier",
    title_fontsize=LEG_FS,
)

# ══════════════════════════════════════════════════════════════════
#  SAVE
# ══════════════════════════════════════════════════════════════════

for path in (OUTPUT_PDF, OUTPUT_PNG):
    fig.savefig(path, bbox_inches="tight", facecolor="white", dpi=300)
    print(f"Saved: {path}")

plt.close(fig)

Saved: figure/fig8_radar.pdf
Saved: figure/fig8_radar.png


In [11]:
"""
generate_tactic_barchart.py
----------------------------
Generates the attack paths / CVEs / CWEs per ATT&CK tactic grouped bar chart.
Uses hatch patterns instead of colors — readable in black and white print.

TUNING PARAMETERS:
    DATASET       — path to your Excel file
    FIGSIZE       — (width, height) in inches
    BAR_WIDTH     — width of each individual bar (0.0–1.0)
    LABEL_FS      — font size of value labels above bars
    TICK_FS       — font size of x-axis tactic labels
    AXIS_LABEL_FS — font size of y-axis label
    TITLE_FS      — font size of chart title
    LEGEND_FS     — font size of legend entries
    Y_PADDING     — extra space above tallest bar
    LABEL_OFFSET  — vertical gap between bar top and label

Usage:
    python generate_tactic_barchart.py

Output:
    figure/Figure_10.pdf
    figure/Figure_10.png

Requirements:
    pip install pandas openpyxl matplotlib numpy
"""

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import rcParams

warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════════
#  TUNING
# ══════════════════════════════════════════════════════════════════

DATASET       = "device_compromise_dataset.xlsx"

FIGSIZE       = (14, 6.5)
BAR_WIDTH     = 0.26
LABEL_FS      = 13
TICK_FS       = 15
AXIS_LABEL_FS = 20
TITLE_FS      = 15
LEGEND_FS     = 14          # legend font size — increased
Y_PADDING     = 3.5
LABEL_OFFSET  = 0.2
LABEL_COLOR   = "#1a1a1a"

# Bar fill: light gray shades that differ in B&W print
COLOR_PATHS   = "#4a4a4a"   # dark gray   — attack paths
COLOR_CVES    = "#909090"   # mid gray    — CVEs
COLOR_CWES    = "#d0d0d0"   # light gray  — CWEs

# Hatch patterns — the main differentiator in B&W
HATCH_PATHS   = "///"       # dense diagonal lines — attack paths
HATCH_CVES    = "xxx"       # crosshatch           — CVEs
HATCH_CWES    = "..."       # dots                 — CWEs

OUTPUT_PDF    = "figure/Figure_10.pdf"
OUTPUT_PNG    = "figure/Figure_10.png"

TACTIC_ORDER = [
    "Initial Access",
    "Credential Access",
    "Defense Evasion",
    "Privilege Escalation",
    "Persistence",
    "Discovery",
    "Lateral Movement",
    "Collection",
]

# ══════════════════════════════════════════════════════════════════
#  STYLE
# ══════════════════════════════════════════════════════════════════

rcParams.update({
    "font.family":       "serif",
    "font.serif":        ["Times New Roman", "DejaVu Serif"],
    "font.size":         15,
    "axes.titlesize":    TITLE_FS,
    "axes.titleweight":  "bold",
    "axes.labelsize":    AXIS_LABEL_FS,
    "xtick.labelsize":   TICK_FS,
    "ytick.labelsize":   TICK_FS,
    "legend.fontsize":   LEGEND_FS,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.grid":         True,
    "grid.alpha":        0.3,
    "grid.linestyle":    "--",
    "axes.linewidth":    0.8,
    "figure.dpi":        150,
    "savefig.dpi":       300,
    "savefig.bbox":      "tight",
    "hatch.linewidth":   0.6,   # controls thickness of hatch lines
})

# ══════════════════════════════════════════════════════════════════
#  DATA
# ══════════════════════════════════════════════════════════════════

df = pd.read_excel(DATASET, index_col=False)
df = df.fillna("")
df["tactics"] = df["tactics"].astype(str).str.strip()

paths_df = df[df["tactics"] != ""].drop_duplicates(
    ["tactics", "cwe_Name", "technique_name", "subtechnique_name", "capec_Name"]
)

tac_cve = df[df["tactics"] != ""].drop_duplicates(["CVE ID", "tactics"])

paths_per_tac = (paths_df.groupby("tactics").size()
                 .reindex(TACTIC_ORDER).fillna(0).astype(int))
cves_per_tac  = (tac_cve.groupby("tactics")["CVE ID"].nunique()
                 .reindex(TACTIC_ORDER).fillna(0).astype(int))
cwes_per_tac  = (paths_df.groupby("tactics")["cwe_Name"].nunique()
                 .reindex(TACTIC_ORDER).fillna(0).astype(int))

# ══════════════════════════════════════════════════════════════════
#  PLOT
# ══════════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=FIGSIZE)

x = np.arange(len(TACTIC_ORDER))

b1 = ax.bar(x - BAR_WIDTH, paths_per_tac.values, width=BAR_WIDTH,
            label="Attack paths",
            color=COLOR_PATHS, hatch=HATCH_PATHS,
            edgecolor="white", linewidth=0.6)

b2 = ax.bar(x,              cves_per_tac.values,  width=BAR_WIDTH,
            label="CVEs",
            color=COLOR_CVES,  hatch=HATCH_CVES,
            edgecolor="white", linewidth=0.6)

b3 = ax.bar(x + BAR_WIDTH,  cwes_per_tac.values,  width=BAR_WIDTH,
            label="CWEs",
            color=COLOR_CWES,  hatch=HATCH_CWES,
            edgecolor="white", linewidth=0.6)

# Value labels above each bar
for bars in (b1, b2, b3):
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                h + LABEL_OFFSET,
                str(int(h)),
                ha="center", va="bottom",
                fontsize=LABEL_FS,
                fontweight="bold",
                color=LABEL_COLOR,
            )

split_labels = [t.replace(" ", "\n") for t in TACTIC_ORDER]
ax.set_xticks(x)
ax.set_xticklabels(split_labels, fontsize=TICK_FS, ha="center")

ax.set_ylabel("Count", fontsize=AXIS_LABEL_FS, labelpad=10)
ax.set_ylim(0, int(paths_per_tac.max()) + Y_PADDING)
ax.set_xlim(-0.65, len(TACTIC_ORDER) - 0.35)
ax.set_axisbelow(True)

ax.set_title(
    "Attack Paths, CVEs, and CWEs per MITRE ATT&CK Tactic",
    loc="left", pad=10, fontsize=TITLE_FS,
)

ax.legend(
    loc="upper right",
    framealpha=0.9,
    handlelength=2.0,   # wider legend patch so hatch pattern is visible
    handleheight=1.6,   # taller legend patch
    borderpad=1.0,
    fontsize=LEGEND_FS,
)

plt.tight_layout(pad=1.5)

# ══════════════════════════════════════════════════════════════════
#  SAVE
# ══════════════════════════════════════════════════════════════════

os.makedirs("figure", exist_ok=True)

for path in (OUTPUT_PDF, OUTPUT_PNG):
    fig.savefig(path, bbox_inches="tight", facecolor="white", dpi=300)
    print(f"Saved: {path}")

plt.close(fig)

Saved: figure/Figure_10.pdf
Saved: figure/Figure_10.png
